# Determines mask for given sulcus

This notebook determines mask around a sulcus. The mask is built in order to only keep sulcus of interest. <br>
It uses a supervised database, in which each sulcus has been manually labelled.

# Imports

In [ ]:
import sys
import os
import glob
import json
import tempfile
import inspect
import zipfile
import dico_toolbox as dtx
import colorado as cld
import pcpm

from urllib.request import urlopen
from soma import aims

In [ ]:
import anatomist.notebook as ana
a = ana.Anatomist()
print(a.headless_info.__dict__)

The following line permits to import deep_folding even if this notebook is executed from the notebooks subfolder (and no install has been launched):

 /notebooks/use_transform.ipynb  
 /deep_folding/__init__.py

In [ ]:
import deep_folding
print(inspect.getfile(deep_folding))

# User-specific variables

In [ ]:
sulcus = 'S.T.s.ter.asc.ant.'

In [ ]:
side = 'L'

In [ ]:
out_voxel_size = 1

We now assign path names and other user-specific variables.

The source directory is where the database lies. It contains the morphologist analysis subfolder ANALYSIS/3T_morphologist


In [ ]:
src_dir = os.path.join(os.getcwd(), '../data/source/supervised')
src_dir = os.path.abspath(src_dir)
print(("src_dir = " + src_dir))

In [ ]:
mask_dir = os.path.join(os.getcwd(), '../data/target/mask')
mask_dir = os.path.abspath(mask_dir)
print(("mask_dir = " + mask_dir))

In [ ]:
ref_dir = os.path.join(os.getcwd(), '../data/reference/mask')
ref_dir = os.path.abspath(ref_dir)
print(("ref_dir = " + ref_dir))

In [ ]:
print((sys.argv))

Gets the normlized SPM file to get voxel size inside the program

norm_dir = os.path.join(os.getcwd(), '../data/source/unsupervised')
norm_dir = os.path.abspath(norm_dir)
sub_dir = "ANALYSIS/3T_morphologist/100206/t1mri/default_acquisition"

# Illustration of main program uses

### Using external calls

In [ ]:
!python ../deep_folding/brainvisa/compute_mask.py --help

### By using the main function call

In [ ]:
from deep_folding.brainvisa import compute_mask
print(compute_mask.__file__)

In [ ]:
args = "--help"
argv = args.split(' ')

In [ ]:
compute_mask.main(argv)

### By using the API function call

In [ ]:
compute_mask.compute_mask(src_dir=src_dir,
                          mask_dir=mask_dir,
                          sulcus=sulcus,
                          side=side,
                          out_voxel_size=out_voxel_size,
                          number_subjects=0)


# Test example

In [ ]:
unsupervised_dir = os.path.join(os.getcwd(), '../data/source/unsupervised')
reference_dir = os.path.join(os.getcwd(), '../data/reference')

In [ ]:
vol_mask = compute_mask.compute_mask(src_dir=src_dir,
                                     mask_dir=mask_dir,
                                     sulcus=sulcus,
                                     side=side,
                                     number_subjects=1,
                                     out_voxel_size=1)


In [ ]:
print(vol_mask.shape) 
assert(vol_mask.shape==(193, 229, 193, 1))

In [ ]:
a_vol_mask = a.toAObject(vol_mask)
axial0 = a.createWindow("Axial")
axial0.addObjects(a_vol_mask)

In [ ]:
temp_dir = tempfile.mkdtemp()
mask_filename_temp = f"{temp_dir}/mask.nii.gz"
aims.write(vol_mask, mask_filename_temp)
bucket_filename = f"{temp_dir}/mask.bck"
cmd = f"AimsFileConvert -c Bucket -t VOID -e 1 -i {mask_filename_temp} -o {bucket_filename}"
os.system(cmd)

# Displays bucket file
bucket, bucket_raw, dxyz, rot, tr = pcpm.load_bucket(bucket_filename)
m = dtx.convert.bucket_to_mesh(bucket)
cld.draw(m)


In [ ]:
cld.draw(bucket_raw)

# Mask test with more than 1 subject

This takes a supervised database outside the deep_folding/data folder

In [ ]:
print(mask_dir)

Note that the following cell will work only at Neurospin.
Otherwise, you just need to chane 'src_dir' to a path to another manually labelled database

In [ ]:
src_dir = "/neurospin/dico/data/bv_databases/human/pclean/all"
mask_dir_temp = f"{temp_dir}/mask"

vol_mask = compute_mask.compute_mask(src_dir=src_dir,
                                     mask_dir=mask_dir_temp,
                                     sulcus=sulcus,
                                     side=side,
                                     number_subjects=10,
                                     out_voxel_size=out_voxel_size)


In [ ]:
c = aims.Converter_rc_ptr_Volume_S16_BucketMap_VOID()
bucket = c(vol_mask)

In [ ]:
# Displays bucket file
m = dtx.convert.bucket_to_mesh(bucket[0])
cld.draw(m)

We now represent the mask together with the MNI template:

In [ ]:
# We recover the MNI template
install_dir = "."
extracted_dir = f"{install_dir}/mni_icbm152_nlin_asym_09c"
if os.path.exists(extracted_dir):
    print(f'the directory {extracted_dir} already exists. Assuming it is OK.')
else:
    dl_url = "http://www.bic.mni.mcgill.ca/~vfonov/icbm/2009/mni_icbm152_nlin_asym_09c_nifti.zip"
    tmp_dl = tempfile.mkstemp(suffix='.zip')
    with urlopen(dl_url) as f:
        with open(tmp_dl[1], 'wb') as g:
            g.write(f.read())
    # Extract the archive
    with zipfile.ZipFile(tmp_dl[1], 'r') as zf:
        zf.extractall(install_dir)

In [ ]:
mni_file = f"{extracted_dir}/mni_icbm152_t1_tal_nlin_asym_09c.nii"
mni = a.loadObject(mni_file)

In [ ]:
mask_vol_aims = vol_mask
print(mask_vol_aims.header())

In [ ]:
# fusion 2D
mask_vol = a.toAObject(mask_vol_aims)
fusion2d = a.fusionObjects([mni, mask_vol], "Fusion2DMethod")
axial = a.createWindow("Axial")
axial.addObjects(fusion2d)
# params of the fusion : linear on non null
a.execute("Fusion2DParams", object=fusion2d, mode="linear_on_defined", rate=0.4)

# Result analysis

We here compare the results with a reference result made with one subject taken from the data unsupervised source folder

Prints the list of files of the target directory

In [ ]:
mask_dir_side = os.path.join(mask_dir, side)
print(mask_dir_side)
print(('\n'.join(os.listdir(mask_dir_side))))

In [ ]:
ref_mask_dir_side = os.path.join(ref_dir, side)
print(ref_mask_dir_side)
list_files = '\n'.join(os.listdir(ref_mask_dir_side))
print(list_files)

In [ ]:
mask_vol_ref = aims.read(glob.glob(f"{ref_mask_dir_side}/*.nii.gz")[0])

In [ ]:
# fusion 2D
mask_ref = a.toAObject(mask_vol_ref)
fusion2d = a.fusionObjects([mask_ref, mask_vol], "Fusion2DMethod")
axial = a.createWindow("Axial")
axial.addObjects(fusion2d)
# params of the fusion : linear on non null
a.execute("Fusion2DParams", object=fusion2d, mode="linear_on_defined", rate=0.4)